# So sánh Feature Selection: Boruta vs Lasso — Life Expectancy

Tương tự `test_housing_lassovsboruta.ipynb` nhưng cho bộ **WHO Life Expectancy**
(`life_expectancy_processed.csv`, target = `Life expectancy`).
So sánh R² của: **baseline (Gradient Boosting, full feature)** vs **Boruta (RF)** vs **Lasso**.

In [1]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from boruta import BorutaPy

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/AIO-Conquer02')
os.getcwd()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/AIO-Conquer02'

In [3]:
import pandas as pd
df = pd.read_csv('life_expectancy_processed.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2928 entries, 0 to 2927
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2928 non-null   object 
 1   Year                             2928 non-null   int64  
 2   Status                           2928 non-null   object 
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2928 non-null   int64  
 6   Alcohol                          2928 non-null   float64
 7   percentage expenditure           2928 non-null   float64
 8   Hepatitis B                      2928 non-null   float64
 9   Measles                          2928 non-null   int64  
 10  BMI                              2928 non-null   float64
 11  under-five deaths                2928 non-null   int64  
 12  Polio               

### Chuẩn bị dữ liệu — encode 2 cột phân loại

Khác bộ housing: ở đây còn 2 cột **chữ** cần xử lý trước khi đưa vào model số:
- **`Status`** (Developing/Developed) → nhị phân **0/1**.
- **`Country`** (193 mức) → **bỏ** trong thí nghiệm này (one-hot sẽ tạo 193 cột rời rạc lấn át; label-encode thì tạo thứ tự giả).
  *Thay thế nếu muốn giữ vị trí:* target/frequency encoding (fit-trên-train).

Target `Life expectancy` giữ **nguyên gốc**.

In [4]:
target_colum = 'Life expectancy'

# encode Status -> 0/1 ; bỏ Country (cardinality cao, xem note ở trên)
df_model = df.copy()
df_model['Status'] = df_model['Status'].map({'Developing': 0, 'Developed': 1})
df_model = df_model.drop(columns=['Country'])

X = df_model.drop(columns=[target_colum], errors='ignore')
y = df_model[target_colum]

# Chia Train/Test trước để tránh data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Số feature:', X.shape[1]); print(X.columns.tolist())

Số feature: 20
['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']


## Baseline — Gradient Boosting với toàn bộ feature

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score  # Dùng R2-score thay cho F1-score

# 1. Khởi tạo mô hình hồi quy Gradient Boosting
gbr = GradientBoostingRegressor(max_depth=5, random_state=42)

# 2. Huấn luyện mô hình
gbr.fit(X_train, y_train)

# 3. Dự đoán tuổi thọ
preds = gbr.predict(X_test)

# 4. Đánh giá bằng R2-score (càng gần 1 càng tốt)
r2_score_all = round(r2_score(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")

Chỉ số R2-score khi sử dụng tất cả feature: 0.962


## Boruta (lõi Random Forest)

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from boruta import BorutaPy

# --- Bước 1: Lấy danh sách tên thuộc tính ban đầu ---
feature_names = X.columns.tolist()

# --- Bước 2: Chuyển sang Numpy để tránh lỗi Index ---
X_train_np = X_train.values if hasattr(X_train, 'columns') else X_train
X_test_np = X_test.values if hasattr(X_test, 'columns') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# --- Bước 3: Mô hình lõi ---
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# --- Bước 4: Cấu hình & chạy Boruta ---
boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=0,
    random_state=42,
    max_iter=100
)
boruta_selector.fit(X_train_np, y_train_np)

# --- Bước 5: Lọc feature được chọn ---
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test_np)

# --- Bước 6: Huấn luyện lại trên feature đã lọc ---
rf_final = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
rf_final.fit(X_train_filtered, y_train_np)
preds_boruta = rf_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test_np, preds_boruta), 3)

confirmed_features = [feature_names[i] for i, c in enumerate(boruta_selector.support_) if c]

# --- Bước 7: In kết quả ---
print("="*60)
print(f"Chỉ số R2-score khi sử dụng Boruta: {r2_score_boruta}")
print(f"Số lượng tính năng được giữ lại: {len(confirmed_features)} / {len(feature_names)}")
print(f"Các tính năng được chọn: {confirmed_features}")
print("="*60)

Chỉ số R2-score khi sử dụng Boruta: 0.934
Số lượng tính năng được giữ lại: 19 / 20
Các tính năng được chọn: ['Year', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']


## Lasso (LassoCV, có chuẩn hoá)

In [7]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# --- Bước 1: Chuẩn hoá (Lasso cần cùng thang đo để phạt công bằng) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bước 2: LassoCV tự tìm alpha tối ưu ---
lasso = LassoCV(
    alphas=np.logspace(-4, 1, 100),
    cv=5,
    max_iter=10000,
    tol=0.01,
    random_state=42,
    n_jobs=-1
)
lasso.fit(X_train_scaled, y_train)

# --- Bước 3: Dự đoán & R2 ---
preds_lasso = lasso.predict(X_test_scaled)
r2_score_lasso = round(r2_score(y_test, preds_lasso), 3)

# --- Bước 4: Feature bị Lasso loại (hệ số = 0) ---
feature_names = X.columns if hasattr(X, 'columns') else [f"Feature_{i}" for i in range(X.shape[1])]
coefficients = lasso.coef_
kept_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef != 0]
dropped_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef == 0]

# --- Bước 5: So sánh với baseline ---
print("="*60)
print(f"[-] R2-score Baseline (Gradient Boosting): {r2_score_all}")
print(f"[+] R2-score khi sử dụng Lasso: {r2_score_lasso}")
print(f" Alpha tối ưu: {round(lasso.alpha_, 5)}")
print("="*60)
print(f" Số feature bị Lasso loại (hệ số = 0): {len(dropped_features)} / {len(feature_names)}")
if len(dropped_features) > 0:
    print(f"   Bị loại: {dropped_features}")
print(f"\n Số feature giữ lại: {len(kept_features)}")
print(f"   Giữ lại: {kept_features}")
print("="*60)

[-] R2-score Baseline (Gradient Boosting): 0.962
[+] R2-score khi sử dụng Lasso: 0.82
 Alpha tối ưu: 0.0026
 Số feature bị Lasso loại (hệ số = 0): 0 / 20

 Số feature giữ lại: 20
   Giữ lại: ['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']
